# 01 · Bronze and Silver
Extracts three public ad datasets from the Kaggle API, lands them in a date-partitioned bronze volume,
loads raw bronze Delta tables, then builds typed and flagged silver tables per source and one unified silver table.

**Run order note:** the finance billing sections at the end depend on `00_simulate_finance_billing`,
which itself needs the silver layer. Run the platform sections first, then `00`, then the billing sections.

## 1. Setup
The Kaggle API token is stored as a file in a Unity Catalog volume and read at runtime. It is never written in code.

In [ ]:
%pip install --upgrade kaggle

In [ ]:
dbutils.library.restartPython()

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.media_pipeline;
CREATE VOLUME IF NOT EXISTS workspace.media_pipeline.config;
CREATE VOLUME IF NOT EXISTS workspace.media_pipeline.bronze;

In [ ]:
import os

# The environment variable must be set before the Kaggle library is imported
with open("/Volumes/workspace/media_pipeline/config/access_token.txt") as f:
    os.environ["KAGGLE_API_TOKEN"] = f.read().strip()

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

## 2. Extraction to the bronze volume
- Files download to a temporary folder and are copied to bronze only on success
- One failing source does not stop the others (error isolation)
- A manifest records files, row counts and status for every run
- Rerunning on the same day overwrites the same files (idempotent)

In [ ]:
import csv, json, shutil, tempfile, logging
from datetime import date, datetime
from pathlib import Path

SOURCES = {
    "social_platform": "loveall/clicks-conversion-tracking",
    "daily_campaign_platform": "amirmotefaker/ab-testing-dataset",
    "multichannel_platform": "manishabhatt22/marketing-campaign-performance-dataset",
}
BRONZE = Path("/Volumes/workspace/media_pipeline/bronze")
ingest_date = date.today().isoformat()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
log = logging.getLogger("extract")

def count_rows(p):
    with open(p, newline="", encoding="utf-8", errors="replace") as f:
        return max(sum(1 for _ in csv.reader(f)) - 1, 0)

results = []
for platform, slug in SOURCES.items():
    try:
        with tempfile.TemporaryDirectory() as tmp:
            api.dataset_download_files(slug, path=tmp, unzip=True, quiet=True)
            target = BRONZE / platform / f"ingest_date={ingest_date}"
            target.mkdir(parents=True, exist_ok=True)
            files = []
            for src in Path(tmp).rglob("*.csv"):
                shutil.copy(src, target / src.name)
                files.append({"file": src.name, "rows": count_rows(src)})
        if not files:
            raise RuntimeError("no CSV files in download")
        log.info("%s: %s", platform, files)
        results.append({"platform": platform, "slug": slug, "status": "success", "files": files})
    except Exception as e:
        log.error("%s failed: %s", platform, e)
        results.append({"platform": platform, "slug": slug, "status": "failed", "error": str(e)})

manifest_dir = BRONZE / "_manifests"
manifest_dir.mkdir(exist_ok=True)
(manifest_dir / f"manifest_{ingest_date}.json").write_text(json.dumps(
    {"ingest_date": ingest_date, "run_at": datetime.now().isoformat(timespec="seconds"), "sources": results},
    indent=2))
print(json.dumps(results, indent=2))

## 3. Profiling
Inspect raw data before writing any cleaning rules. Everything is read as text so no problem is hidden by type guessing.

Findings that drive the silver rules: semicolon delimiter and `d.MM.yyyy` dates in the daily source;
one partial record (5 Aug 2019); funnel anomalies; 204 zero-click conversions in social (view-through);
`$16,174.00`-style cost and `30 days`-style duration in multichannel; no dates in social.

In [ ]:
from pyspark.sql import functions as F

BRONZE_PATH = "/Volumes/workspace/media_pipeline/bronze"

daily = (spark.read.option("header", True).option("sep", ";")
         .csv(f"{BRONZE_PATH}/daily_campaign_platform/*/*.csv")
         .withColumn("source_file", F.col("_metadata.file_name")))
display(daily.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in daily.columns]))

social = spark.read.option("header", True).csv(f"{BRONZE_PATH}/social_platform/*/*.csv")
print("Conversions with zero clicks:",
      social.filter((F.col("Clicks").cast("int") == 0) & (F.col("Total_Conversion").cast("int") > 0)).count())
print("Approved > Total:",
      social.filter(F.col("Approved_Conversion").cast("int") > F.col("Total_Conversion").cast("int")).count())

multi = spark.read.option("header", True).csv(f"{BRONZE_PATH}/multichannel_platform/*/*.csv")
display(multi.select("Acquisition_Cost", "Duration", "Conversion_Rate", "ROI").limit(10))

## 4. Bronze Delta tables
Raw data, untouched, all text. Only a technical rename of characters Delta does not allow in column names.
`_source_file` and `_ingested_at` record lineage.

In [ ]:
import re

SOURCE_FILES = {
    "bronze_social":       (f"{BRONZE_PATH}/social_platform/*/*.csv", ","),
    "bronze_daily":        (f"{BRONZE_PATH}/daily_campaign_platform/*/*.csv", ";"),
    "bronze_multichannel": (f"{BRONZE_PATH}/multichannel_platform/*/*.csv", ","),
}

def safe_name(col):
    return re.sub(r"[^0-9a-zA-Z]+", "_", col).strip("_")

for table, (path, sep) in SOURCE_FILES.items():
    df = (spark.read.option("header", True).option("sep", sep).csv(path)
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingested_at", F.current_timestamp()))
    df = df.toDF(*[c if c.startswith("_") else safe_name(c) for c in df.columns])
    df.write.mode("overwrite").saveAsTable(f"workspace.media_pipeline.{table}")
    print(f"{table}: {spark.table(f'workspace.media_pipeline.{table}').count()} rows")

## 5. Silver: daily campaign platform
`CAST` (not `try_cast`) so unexpected values fail loudly. `CREATE OR REPLACE ... AS SELECT` keeps it idempotent.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.silver_daily AS
SELECT
  'daily_campaign_platform'                 AS platform,
  Campaign_Name                             AS campaign_name,
  to_date(`Date`, 'd.MM.yyyy')              AS report_date,
  CAST(Spend_USD AS DECIMAL(12,2))          AS spend,
  CAST(of_Impressions AS INT)               AS impressions,
  CAST(Reach AS INT)                        AS reach,
  CAST(of_Website_Clicks AS INT)            AS clicks,
  CAST(of_Searches AS INT)                  AS searches,
  CAST(of_View_Content AS INT)              AS view_content,
  CAST(of_Add_to_Cart AS INT)               AS add_to_cart,
  CAST(of_Purchase AS INT)                  AS purchases,
  of_Impressions IS NULL                    AS is_incomplete,
  COALESCE(CAST(of_Purchase AS INT) > CAST(of_Add_to_Cart AS INT)
        OR CAST(of_Add_to_Cart AS INT) > CAST(of_View_Content AS INT), FALSE)
                                            AS has_funnel_anomaly,
  _source_file
FROM workspace.media_pipeline.bronze_daily;

In [ ]:
%sql
-- Expected: 60 rows, 0 bad dates, 1 incomplete row, 9 funnel anomalies, 2019-08-01 to 2019-08-30
SELECT
  COUNT(*)                        AS total_rows,
  COUNT_IF(report_date IS NULL)   AS bad_dates,
  COUNT_IF(is_incomplete)         AS incomplete_rows,
  COUNT_IF(has_funnel_anomaly)    AS funnel_anomalies,
  MIN(report_date)                AS first_day,
  MAX(report_date)                AS last_day
FROM workspace.media_pipeline.silver_daily;

## 6. Silver: social platform
Zero-click conversions are kept as view-through conversions, not deleted. The source has no dates.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.silver_social AS
SELECT
  'social_platform'                                              AS platform,
  ad_id                                                          AS ad_id,
  xyz_campaign_id                                                AS campaign_id,
  fb_campaign_id                                                 AS platform_campaign_id,
  age                                                            AS age_group,
  gender                                                         AS gender,
  CAST(interest AS INT)                                          AS interest_id,
  CAST(Impressions AS INT)                                       AS impressions,
  CAST(Clicks AS INT)                                            AS clicks,
  CAST(Spent AS DECIMAL(12,2))                                   AS spend,
  CAST(Total_Conversion AS INT)                                  AS conversions,
  CAST(Approved_Conversion AS INT)                               AS approved_conversions,
  CAST(Clicks AS INT) = 0 AND CAST(Total_Conversion AS INT) > 0  AS is_view_through,
  _source_file
FROM workspace.media_pipeline.bronze_social;

In [ ]:
%sql
-- Expected: 1143 rows, 204 view-through rows, social_platform, int
SELECT
  COUNT(*)                     AS total_rows,
  COUNT_IF(is_view_through)    AS view_through_rows,
  MIN(platform)                AS platform_value,
  typeof(MIN(clicks))          AS clicks_type
FROM workspace.media_pipeline.silver_social;

## 7. Silver: multichannel platform
Cost text like `$16,174.00` is cleaned before conversion. `Acquisition_Cost` is treated as total campaign spend (documented assumption).

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.silver_multichannel AS
SELECT
  'multichannel_platform'                                                       AS platform,
  Campaign_ID                                                                   AS campaign_id,
  Company                                                                       AS company,
  Campaign_Type                                                                 AS campaign_type,
  Target_Audience                                                               AS target_audience,
  Channel_Used                                                                  AS channel,
  CAST(REPLACE(Duration, ' days', '') AS INT)                                   AS duration_days,
  to_date(`Date`, 'yyyy-MM-dd')                                                 AS report_date,
  CAST(REPLACE(REPLACE(Acquisition_Cost, '$', ''), ',', '') AS DECIMAL(12,2))   AS spend,
  CAST(Impressions AS INT)                                                      AS impressions,
  CAST(Clicks AS INT)                                                           AS clicks,
  CAST(Conversion_Rate AS DECIMAL(5,4))                                         AS conversion_rate,
  CAST(ROI AS DECIMAL(6,2))                                                     AS roi,
  CAST(Engagement_Score AS INT)                                                 AS engagement_score,
  Location                                                                      AS location,
  Language                                                                      AS language,
  Customer_Segment                                                              AS customer_segment,
  _source_file
FROM workspace.media_pipeline.bronze_multichannel;

In [ ]:
%sql
-- Expected: 200000 rows, zero nulls, 2021-01-01 to 2021-12-31, spend 5000 to 20000
SELECT
  COUNT(*)                           AS total_rows,
  COUNT_IF(spend IS NULL)            AS null_spend,
  COUNT_IF(duration_days IS NULL)    AS null_duration,
  COUNT_IF(report_date IS NULL)      AS null_dates,
  MIN(report_date)                   AS first_day,
  MAX(report_date)                   AS last_day,
  MIN(spend)                         AS min_spend,
  MAX(spend)                         AS max_spend
FROM workspace.media_pipeline.silver_multichannel;

## 8. Silver: unified campaign performance
`UNION ALL` (never `UNION`, which would silently remove identical rows). Same columns in the same order in every `SELECT`,
with typed nulls so the result does not depend on the order of the `SELECT`s.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.silver_campaign_performance AS
SELECT
  platform,
  campaign_name             AS campaign_id,
  report_date,
  spend,
  impressions,
  clicks,
  purchases                 AS conversions,
  is_incomplete,
  FALSE                     AS is_view_through,
  has_funnel_anomaly,
  _source_file
FROM workspace.media_pipeline.silver_daily

UNION ALL

SELECT
  platform,
  campaign_id,
  CAST(NULL AS DATE)        AS report_date,
  spend,
  impressions,
  clicks,
  conversions,
  FALSE                     AS is_incomplete,
  is_view_through,
  FALSE                     AS has_funnel_anomaly,
  _source_file
FROM workspace.media_pipeline.silver_social

UNION ALL

SELECT
  platform,
  campaign_id,
  report_date,
  spend,
  impressions,
  clicks,
  CAST(NULL AS INT)         AS conversions,
  FALSE                     AS is_incomplete,
  FALSE                     AS is_view_through,
  FALSE                     AS has_funnel_anomaly,
  _source_file
FROM workspace.media_pipeline.silver_multichannel;

In [ ]:
%sql
-- Reconciliation: expected 60 / 1143 / 200000 rows, and spend matching each source silver table to the cent
SELECT platform, COUNT(*) AS rows, SUM(spend) AS total_spend
FROM workspace.media_pipeline.silver_campaign_performance
GROUP BY platform
ORDER BY platform;

In [ ]:
%sql
SELECT 'daily' AS src, SUM(spend) AS total_spend FROM workspace.media_pipeline.silver_daily
UNION ALL
SELECT 'social', SUM(spend) FROM workspace.media_pipeline.silver_social
UNION ALL
SELECT 'multichannel', SUM(spend) FROM workspace.media_pipeline.silver_multichannel;

## 9. Finance billing (run after `00_simulate_finance_billing`)
The billing feed is treated like any external source. Duplicate invoices are kept and flagged, not deleted.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.bronze_finance_billing AS
SELECT *,
       _metadata.file_path  AS _source_file,
       current_timestamp()  AS _ingested_at
FROM read_files(
  '/Volumes/workspace/media_pipeline/bronze/finance_billing/*/*.csv',
  format => 'csv', header => true, inferColumnTypes => false
);

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.silver_finance_billing AS
SELECT
  invoice_id,
  vendor                                        AS platform,
  billing_month,
  CAST(amount AS DECIMAL(12,2))                 AS amount,
  currency,
  COUNT(*) OVER (PARTITION BY invoice_id) > 1   AS is_duplicate_invoice,
  _source_file
FROM workspace.media_pipeline.bronze_finance_billing;

In [ ]:
%sql
-- Expected: 15 invoice lines, 2 duplicate lines
SELECT COUNT(*)                       AS invoice_lines,
       COUNT_IF(is_duplicate_invoice) AS duplicate_lines,
       SUM(amount)                    AS total_billed
FROM workspace.media_pipeline.silver_finance_billing;